# Updaters

<style>
blockquote:has(.notebook-admonition-title) {
  --notebook-admonition-color: var(--color-admonition-title--note, #087fc7);
  --notebook-admonition-title-background:
    var(--color-admonition-title-background--note, rgba(8, 127, 199, 0.18));
  background: var(--color-admonition-background, transparent);
  border: 0;
  border-left: 0.2rem solid var(--notebook-admonition-color);
  border-radius: 0.2rem;
  box-shadow: 0 0.2rem 0.5rem rgba(0, 0, 0, 0.05), 0 0 0.0625rem rgba(0, 0, 0, 0.1);
  font-size: var(--admonition-font-size, 0.8125rem);
  margin: 1rem auto;
  overflow: hidden;
  padding: 0 0.5rem 0.5rem;
}
blockquote:has(.notebook-admonition-title.important) {
  --notebook-admonition-color:
    var(--color-admonition-title--attention, #ff5252);
  --notebook-admonition-title-background:
    var(--color-admonition-title-background--attention, rgba(255, 82, 82, 0.2));
}
blockquote p:has(> .notebook-admonition-title) {
  background: var(--notebook-admonition-title-background);
  font-size: var(--admonition-title-font-size, 0.8125rem);
  font-weight: 500;
  line-height: 1.3;
  margin: 0 -0.5rem 0.5rem;
  padding: 0.4rem 0.5rem 0.4rem 2rem;
  position: relative;
}
blockquote p:has(> .notebook-admonition-title)::before {
  color: var(--notebook-admonition-color);
  content: "✎";
  left: 0.65rem;
  position: absolute;
}
blockquote p:has(> .notebook-admonition-title.important)::before {
  content: "⚠";
}
.notebook-admonition-title {
  font-weight: inherit;
}
</style>

Depending on the questions you are investigating, there are many different
values you might want to compute for each partition in your Markov chain. If you
are interested in compactness, you might want to compute the area and perimeter
of each part of the partition so that you can compute compactness scores. If you
are interested in partisan lean, you might want to compute hypothetical election
results using the districts defined by the partition.

The `Partition` class allows you to define custom properties for the partitions
in your Markov chain. You can do this by providing a dictionary of updater
functions when you first create a partition.

In [ ]:
import networkx
from gerrychain import Partition, Graph

# Use NetworkX to create a graph
nx_graph = networkx.Graph()
nx_graph.add_edges_from([(0, 1), (1, 2), (2, 0)])

# Create a GerryChain Graph object from the NetworkX Graph object
graph = Graph.from_networkx(nx_graph)

assignment = {0: 1, 1: 1, 2: 2}


def my_updater(partition):
    return "Hello!"


partition = Partition(graph, assignment, {"my_custom_property": my_updater})

print(partition["my_custom_property"])

This partition and all subsequent partitions in the chain will have this
`my_custom_property` attribute. If we flip a node in `partition` to create a new
partition, we can still access this property:

In [ ]:
new_partition = partition.flip({1: 2})
print(f"Are the partitions different? {new_partition is not partition}")
print(new_partition["my_custom_property"])

## Useful updater functions in GerryChain

The `gerrychain.updaters` submodule provides some updaters for common tasks like
aggregating data and computing the cut edges of a partition:

- `Tally`: Aggregates a node attribute (e.g. population) over each part of the
  partition.
- `cut_edges`: Returns the set of cut edges (edges whose nodes are in
  different parts of the partition) of the partition. This is required for
  most of the proposal functions in `gerrychain.proposals`.

Here is an example using both of these updaters:

In [ ]:
from gerrychain.updaters import cut_edges, Tally

# Use NetworkX to create a 2x2 grid graph
nx_graph = networkx.Graph()
nx_graph.add_edges_from([(0, 1), (1, 2), (2, 3), (3, 0)])

# Create a GerryChain Graph object from the NetworkX Graph object
graph = Graph.from_networkx(nx_graph)

# Give each of the nodes population 100:
for node_id in graph.node_indices:
    graph.node_data(node_id)["population"] = 100

# Partition the grid into two halves:
assignment = {0: 0, 1: 0, 2: 1, 3: 1}
partition = Partition(
    graph, assignment, updaters={"cut_edges": cut_edges, "population": Tally("population")}
)
print(f"Population by district:\n\t{partition['population']}")
print(f"Cut edges in partition:\n\t{partition['cut_edges']}")

Our `cut_edges` updater returns a set of edges, each represented as a tuple of
two nodes. Our `population` updater returns a dictionary mapping each part of
the partition to the total population in that part. Since we divided our grid in
half, we see parts `0` and `1` both have population 200.

Now when we create a new partition by flipping a node of `partition`, we see the
values of the updaters change:

In [ ]:
new_partition = partition.flip({0: 1})
print(f"Population by district:\n\t{new_partition['population']}")
print(f"Cut edges in partition:\n\t{new_partition['cut_edges']}")

As we should expect, flipping node `0` into part `1` increases the population of
part `1` to 300 and decreases the population of part `0` to 100. The cut edges
of the new partition are both of the edges incident to node `1`, since this is
the last remaining node in part `0`.

## Writing your own updater function

When using GerryChain to experiment with new metrics, proposals, or acceptance
rules, there usually comes a point when you need to implement a new updater. As
we saw in the first example, an updater is a function that takes the partition
as its argument and returns any type of value.

Let's create an updater that returns the number of cut edges in the partition.

In [ ]:
def number_of_cut_edges(partition):
    return len(partition["cut_edges"])

> <span class="notebook-admonition-title important">Important</span>
>
> Note that this updater uses the value of the `cut_edges` updater in its
> computation. This is completely allowed! All you need to do is make sure that
> any updater that your updater depends on is included in the `updaters`
> dictionary that we pass to `Partition`. We also need to make sure that we have
> no cyclic dependencies: if the `cut_edges` updater also depended on
> `number_of_cut_edges`, we would fall into an infinite loop when we called either
> of them, resulting in a `RuntimeError: maximum recursion depth exceeded` error.

To try out our updater, we'll use [NetworkX](https://networkx.github.io) to
create a complete graph on 4 nodes, which we'll partition in halves like the 2x2
grid. NetworkX is the graph library that GerryChain uses under the hood in its
`Graph` class.

In [ ]:
import networkx

nx_graph = networkx.complete_graph(4)
graph = Graph.from_networkx(nx_graph)
assignment = {0: 0, 1: 0, 2: 1, 3: 1}
my_updaters = {"cut_edges": cut_edges, "number_of_cut_edges": number_of_cut_edges}
partition = Partition(graph, assignment, my_updaters)

Now we can try out our custom `number_of_cut_edges` updater, and verify that its
value changes when the partition changes:

In [ ]:
print(partition["number_of_cut_edges"])
new_partition = partition.flip({0: 1})
print(new_partition["number_of_cut_edges"])